### Objective
The objective is to build a classification model to predict whether a Formula 1 driver will finish in the top 3 (Top3 = 1) or not (Top3 = 0) based on pre-race features such as grid position, driver, constructor, and race characteristics.


In [37]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

### Data Preparation
First, the two datasets (results.csv and final_dataset.csv) were merged using raceId to combine driver-level and race-level information.

The target variable Top3 was created:
Top3 = 1 if position ≤ 3
Top3 = 0 otherwise
Missing values in position were removed to ensure correct labeling.
Features used:
Numerical: grid, year, round
Categorical: driverId, constructorId, circuitId, country
Categorical variables were converted using One-Hot Encoding, since SVC cannot handle categorical data directly.
Numerical features were scaled using StandardScaler, because SVC is sensitive to feature magnitudes.
The dataset was split into training and testing sets using an 80–20 split with stratification to preserve class balance

In [18]:
# Load the dataset
results = pd.read_csv("./results.csv")
final_data = pd.read_csv("./final_dataset.csv")

In [20]:
# Keep only needed columns
results_df = results[['raceId', 'driverId', 'constructorId', 'grid', 'position']]
final_df = final_data[['raceId', 'year', 'round', 'circuitId', 'country']]

In [21]:
# Merge datasets
df = pd.merge(results_df, final_df, on='raceId', how='inner')

In [14]:
df.head()

,driverId,constructorId,grid,year,round,circuitId,country,alt,lat,lng,Top3
0,20,9,1,2011,1,1,Australia,10,-37.8497,144.968,1
1,1,1,2,2011,1,1,Australia,10,-37.8497,144.968,1
2,808,4,6,2011,1,1,Australia,10,-37.8497,144.968,1
3,4,6,5,2011,1,1,Australia,10,-37.8497,144.968,0
4,17,9,3,2011,1,1,Australia,10,-37.8497,144.968,0


In [15]:
df.columns

Index(['driverId', 'constructorId', 'grid', 'year', 'round', 'circuitId',
       'country', 'alt', 'lat', 'lng', 'Top3'],
      dtype='object')

In [22]:
# Create target
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df = df.dropna(subset=['position'])
df['Top3'] = (df['position'] <= 3).astype(int)

In [23]:
# Create features and target
X = df[['driverId', 'constructorId', 'grid', 'year', 'round', 'circuitId', 'country']]
y = df['Top3']

In [24]:
# Encode categorical variables
X = pd.get_dummies(X, columns=['driverId', 'constructorId', 'circuitId', 'country'])

In [25]:
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [26]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Feature scaling was applied using StandardScaler to standardize numerical variables. This is important for SVC because it relies on distance calculations, and features with larger values can dominate the model. Scaling ensures all features contribute equally and improves model performance. The scaler was fitted on training data and applied to test data to avoid data leakage.

In [27]:
# Apply support vector classifier
svc_model = SVC()
svc_model.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [30]:
# Hyperparameter tuning
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.1],
    'kernel': ['rbf']}

grid = GridSearchCV(
    SVC(), 
    param_grid,
    cv = 5,
    scoring = 'f1')

grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_

GridSearchCV was used to test different combinations of SVC parameters such as C, gamma, and kernel to find the best model. A 5-fold cross-validation was applied to ensure the model generalizes well. The F1-score was used for evaluation since the data is imbalanced. The best parameter combination was selected as best_model for final predictions.

In [31]:
# Model Evaluation
y_pred = best_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
confusion_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Accuracy: {:.2f}%".format(accuracy*100))
print("Confusion Matrix: \n", confusion_matrix)
print("Classification Report: \n", report)

Accuracy: 87.16%
Confusion Matrix: 
 [[761  67]
 [ 61 108]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.93      0.92      0.92       828
           1       0.62      0.64      0.63       169

    accuracy                           0.87       997
   macro avg       0.77      0.78      0.78       997
weighted avg       0.87      0.87      0.87       997



The SVC model achieved an accuracy of 87.16%, indicating good overall performance in classifying whether a driver finishes in the top 3 or not. From the confusion matrix, the model correctly predicted 761 non-top 3 drivers and 108 top 3 drivers, but it also misclassified 67 non-top 3 drivers as top 3 and missed 61 actual top-3 drivers. The classification report shows strong performance for the non-top 3 class with high precision (0.93) and recall (0.92), while performance for the top-3 class is lower, with precision (0.62) and recall (0.64). This indicates that the model is better at identifying non-top 3 outcomes and struggles more with correctly predicting top 3 finishes. Overall, despite high accuracy, the moderate F1-score for the top 3 class suggests that class imbalance affects performance, making F1-score a more meaningful evaluation metric than accuracy in this problem.

### Compared to tree-based methods

In [39]:
# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, dt_pred))
print("Decision Tree Classification Report:\n", classification_report(y_test, dt_pred))

Decision Tree Accuracy: 0.8976930792377131
Decision Tree Confusion Matrix:
 [[778  50]
 [ 52 117]]
Decision Tree Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.94      0.94       828
           1       0.70      0.69      0.70       169

    accuracy                           0.90       997
   macro avg       0.82      0.82      0.82       997
weighted avg       0.90      0.90      0.90       997



In [40]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=10
)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

print("\nRandom Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Random Forest Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))
print("Random Forest Classification Report:\n", classification_report(y_test, rf_pred))



Random Forest Accuracy: 0.8966900702106319
Random Forest Confusion Matrix:
 [[811  17]
 [ 86  83]]
Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.98      0.94       828
           1       0.83      0.49      0.62       169

    accuracy                           0.90       997
   macro avg       0.87      0.74      0.78       997
weighted avg       0.89      0.90      0.89       997



The tree-based models, including Decision Tree and Random Forest, achieved high overall accuracy of approximately 90%, slightly outperforming the SVC model. The Decision Tree showed balanced performance, with good precision (0.70) and recall (0.69) for predicting top 3 drivers, resulting in an F1-score of 0.70. This indicates that it is relatively effective at identifying top 3 finishes. In contrast, the Random Forest model achieved higher precision (0.83) for the top 3 class but much lower recall (0.49), meaning it correctly identifies fewer actual top 3 drivers while being more conservative in its predictions. Both models perform very well for the non-top 3 class, but their ability to predict top 3 outcomes varies. Compared to SVC, tree based methods achieve slightly higher accuracy, but SVC provides a more balanced trade-off between precision and recall for the top 3 class. Overall, the Decision Tree appears to offer the best balance among the models, while Random Forest prioritizes precision over recall.